# GPT-SoVITS 純推論 server — Colab

v2Pro + 洛神。跑完就會得到一個可以操作的控制台，以及**真實的 GPU 延遲數字**。

> **先確認有 GPU**：上方選單 `執行階段` → `變更執行階段類型` → 硬體加速器選 **T4 GPU**。
> 沒有 GPU 也能跑，但即時倍率只有 0.6–0.8×（達不到實時）。

依序執行下面的 cell 即可。第一次要下載約 2.85 GB 權重。

Repo: https://github.com/jtswwan/tts_model_only_inference

## 1. 檢查執行環境

In [ ]:
import subprocess, sys, platform
print("Python", platform.python_version())
r = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
                    "--format=csv,noheader"], capture_output=True, text=True)
if r.returncode == 0:
    print("GPU:", r.stdout.strip())
else:
    print("⚠ 沒有偵測到 GPU。執行階段 → 變更執行階段類型 → T4 GPU")
    print("  沒有 GPU 也能跑完，但達不到實時。")

## 2. 取得程式碼

In [ ]:
import os, pathlib
REPO_DIR = pathlib.Path("/content/tts_model_only_inference")
if not REPO_DIR.exists():
    !git clone --depth 1 -b claude/blissful-hawking-safc6x https://github.com/jtswwan/tts_model_only_inference {REPO_DIR}
else:
    print("已存在，跳過")
os.chdir(REPO_DIR)
print("工作目錄:", os.getcwd())

## 3. 安裝依賴

Colab 已經預裝 CUDA 版的 torch，重裝要好幾分鐘又佔好幾 GB，所以先檢查能不能沿用。

有一個相容性問題要處理：**torchaudio 從 2.8/2.9 起把自己的 I/O backend 移除**，
`torchaudio.load()` 改委派給 `torchcodec`，而 `TTS.py:773` 讀參考音訊正是用它。
下面會依序試三條路，挑最省的那條：

1. Colab 現成的 torchaudio 就能讀 wav → 什麼都不用動
2. 裝 `torchcodec` 就能讀 → 只加一個套件
3. 都不行 → 才降級 torch/torchaudio 到 `<2.9`（最慢，要重裝 CUDA 版 torch）

In [ ]:
import subprocess, sys, importlib

def sh(*args):
    print("$", " ".join(args))
    return subprocess.run(args).returncode

def can_load_wav() -> bool:
    """實測 torchaudio.load 能不能讀 wav —— 這才是真正要的能力。"""
    probe = '''
import torchaudio, torch, tempfile, os
sr = 32000
wav = torch.zeros(1, sr)          # 1 秒靜音
p = os.path.join(tempfile.mkdtemp(), "probe.wav")
torchaudio.save(p, wav, sr, encoding="PCM_S", bits_per_sample=16)
a, got = torchaudio.load(p)
assert got == sr and a.shape[1] == sr
print("OK")
'''
    r = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
    if r.returncode != 0:
        print("  torchaudio.load 失敗:", (r.stderr or "").strip().splitlines()[-1][:160])
    return r.returncode == 0

import torch, torchaudio
print(f"現成版本: torch {torch.__version__} / torchaudio {torchaudio.__version__}")

if can_load_wav():
    print("✓ 路徑 1：現成的 torchaudio 就能讀 wav，不動它")
else:
    print("→ 路徑 2：試裝 torchcodec")
    sh(sys.executable, "-m", "pip", "install", "-q", "torchcodec")
    if can_load_wav():
        print("✓ 路徑 2 成功")
    else:
        print("→ 路徑 3：降級 torch / torchaudio 到 <2.9（會花幾分鐘）")
        sh(sys.executable, "-m", "pip", "install", "-q",
           "torch>=2.1,<2.9", "torchaudio>=2.1,<2.9",
           "--index-url", "https://download.pytorch.org/whl/cu121")
        print("⚠ 降級後請重新啟動執行階段（執行階段 → 重新啟動），再從這個 cell 繼續")

### 3b. 其餘套件

`requirements.txt` 裡的 torch 已經處理過了，這裡只裝其餘的。

`numpy<2.0` 是 upstream 的釘選。如果 Colab 的 numpy 是 2.x，pip 會降級它 —— 降級後**必須重新啟動執行階段**，
因為已經 import 過的 numpy 不會換掉。下面會偵測並提示。

In [ ]:
import numpy, subprocess, sys
before = numpy.__version__
print("numpy（安裝前）:", before)

# 跳過 torch/torchaudio（上一個 cell 已處理），其餘照 requirements.txt 裝
req = [l.strip() for l in open("requirements.txt", encoding="utf-8")
       if l.strip() and not l.startswith("#")]
req = [l for l in req if not l.split(">")[0].split("=")[0].split("<")[0].strip()
       in ("torch", "torchaudio")]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *req], check=False)

after = subprocess.run([sys.executable, "-c", "import numpy;print(numpy.__version__)"],
                       capture_output=True, text=True).stdout.strip()
print("numpy（安裝後）:", after)
if after != before:
    print()
    print("⚠ numpy 版本變了，必須重新啟動執行階段才會生效：")
    print("   執行階段 → 重新啟動工作階段，然後從『2. 取得程式碼』重新執行")
else:
    print("✓ 不需要重新啟動")

## 4. 下載模型權重（約 2.85 GB）

同時會抓英文前端要的 nltk 語料（cmudict 與 averaged_perceptron_tagger，約 20 MB）到
專案內的 `nltk_data/`。少了它，**含英文或數字的句子會在串流中途失敗**（純中文不受影響）——
因為 `text/english.py` 透過 `g2p_en` 需要這兩份語料。

In [ ]:
!python3 scripts/start.py --download-only

## 5. 啟動 server

在背景啟動，並等暖機完成。第一次會載入模型、跑 BERT 剪裁的等價性自檢、
再跑一次 dummy 推論把一次性成本付掉 —— GPU 上通常不到一分鐘。

In [ ]:
import subprocess, sys, time, json, urllib.request, pathlib, os

PORT = 8000
LOG = pathlib.Path("/content/server.log")
proc = subprocess.Popen(
    [sys.executable, "-m", "gsv_server", "--host", "127.0.0.1", "--port", str(PORT)],
    cwd="server", stdout=open(LOG, "w"), stderr=subprocess.STDOUT)

def health():
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=4) as r:
            return json.load(r)
    except Exception:
        return None

print("暖機中", end="")
t0 = time.time()
info = None
while time.time() - t0 < 900:
    if proc.poll() is not None:
        print("\n✗ server 結束了，log 如下：")
        print(LOG.read_text()[-3000:])
        raise SystemExit(1)
    info = health()
    if info and info.get("ready"):
        break
    print(".", end="", flush=True)
    time.sleep(5)

print(f"\n就緒（{time.time()-t0:.0f}s）")
print(json.dumps(info, ensure_ascii=False, indent=1))
q = info["quality"]
print()
print("音質狀態：", "✓ 與 upstream 預設一致" if q["clean"] else "⚠ " + "；".join(q["deviations"]))
print("BERT 剪裁等價性：", "✓ 逐位元驗證通過" if q["bert_trim_verified"] else "未啟用")

## 6. 打開控制台

Colab 看不到 localhost，要透過它自己的 proxy。下面會開一個新視窗。
如果被瀏覽器擋了，改用第二行的 iframe 版本。

In [ ]:
from google.colab import output
output.serve_kernel_port_as_window(PORT)      # 開新視窗
# output.serve_kernel_port_as_iframe(PORT, height=1400)   # 或嵌在這裡

## 7. 量真實的 GPU 延遲

這是這個 notebook 最有價值的部分 —— **TTFA 的實測值**。

量的是客戶端收到第一個 byte 的時間，不是伺服器自己回報的數字。
「即時倍率」大於 1 表示生成快於播放，串流可以不中斷。

In [ ]:
import time, urllib.parse, urllib.request, statistics, pathlib
from IPython.display import Audio, display

PORT = globals().get("PORT", 8000)   # 重啟後單獨執行這個 cell 也能用

TEXTS = [
    ("短", "你好。"),
    ("中", "慢慢來也沒關係，只要沒有停下來，你其實一直都在前進。"),
    ("長", "明天的天氣預報是：早上會有一點懶洋洋的微風，適合慢慢醒來；"
           "中午陽光剛好，不刺眼，但足夠讓人想出去走走；"
           "下午可能會飄來幾朵心情很好的雲，把天空點綴得剛剛好；"
           "晚上氣溫微涼，適合聽音樂或發呆。"),
    ("中英混合", "這個 server 用 v2Pro 模型，sample rate 是 32000 Hz。"),
]

# 回傳 (TTFA, 總時間, PCM, sample rate)。
# /tts/stream 是 chunked 串流，所以 server 如果在串流「中途」出錯，HTTP 狀態碼
# 早就送出 200 了 —— 客戶端只會拿到 IncompleteRead，看不出原因。這裡把它接住，
# 改成去 server log 撈最後幾行錯誤印出來。
def measure(text):
    url = f"http://127.0.0.1:{PORT}/tts/stream?" + urllib.parse.urlencode({"text": text})
    t0 = time.perf_counter(); first = None; buf = bytearray()
    try:
        with urllib.request.urlopen(url, timeout=600) as r:
            sr = int(r.headers.get("X-Sample-Rate", 32000))
            while True:
                c = r.read(4096)
                if not c: break
                if first is None: first = time.perf_counter() - t0
                buf += c
    except Exception as exc:
        tail = ""
        try:
            lines = pathlib.Path("/content/server.log").read_text(
                encoding="utf-8", errors="replace").strip().splitlines()
            tail = "\n".join("    " + l for l in lines[-12:])
        except Exception:
            pass
        raise RuntimeError(
            f"串流中斷（{type(exc).__name__}: {exc}）。已收到 {len(buf)} bytes。\n"
            f"  server log 末尾：\n{tail}") from exc
    return first, time.perf_counter() - t0, bytes(buf), sr

print("暖身一次（不計入）…"); measure(TEXTS[1][1])
print()
print(f"{'文字':10s} {'字數':>4s}  {'TTFA':>9s} {'總時間':>9s} {'音訊':>8s} {'即時倍率':>9s}")
print("-" * 58)
rows = []
for label, text in TEXTS:
    ttfa, total, pcm, sr = measure(text)
    secs = len(pcm) / 2 / sr
    rows.append((label, text, ttfa, total, secs, pcm, sr))
    print(f"{label:10s} {len(text):4d}  {ttfa*1000:8.0f}ms {total*1000:8.0f}ms "
          f"{secs:7.2f}s {secs/total:8.2f}×")

print()
slow = [r for r in rows if r[4] / r[3] < 1]
if slow:
    print("⚠ 這些長度的即時倍率 < 1，串流會追不上播放：", ", ".join(r[0] for r in slow))
else:
    print("✓ 全部即時倍率 > 1 —— 串流不會中斷")

### 聽聽看

In [ ]:
import struct
from IPython.display import Audio, display

def wav_bytes(pcm, sr):
    head = (b"RIFF" + struct.pack("<I", 36 + len(pcm)) + b"WAVEfmt " +
            struct.pack("<IHHIIHH", 16, 1, 1, sr, sr * 2, 2, 16) +
            b"data" + struct.pack("<I", len(pcm)))
    return head + pcm

for label, text, ttfa, total, secs, pcm, sr in rows:
    print(f"【{label}】TTFA {ttfa*1000:.0f}ms · {secs:.2f}s · {text[:36]}")
    display(Audio(wav_bytes(pcm, sr), rate=sr))

## 8. 換模型 / 換參考音訊（選用）

上傳自己的權重後，用 `add_voice.py` 檢查並安裝。它會從**檔頭**判定版本、
核對 GPT 與 SoVITS 的符號集、檢查參考音訊長度是否在 3–10 秒內。

In [ ]:
# from google.colab import files
# up = files.upload()        # 上傳 .pth / .ckpt / .wav

# !python3 scripts/add_voice.py --dry-run --name 新角色 \
#     --sovits /content/新角色_e8.pth \
#     --gpt    /content/新角色-e15.ckpt \
#     --ref    /content/參考.wav \
#     --ref-text "參考音訊的逐字稿"

# 檢查通過後加 --server http://127.0.0.1:8000 直接套用到執行中的 server

## 9. 停止

Colab 的執行階段結束時 server 會自動結束，通常不需要手動停。

In [ ]:
# proc.terminate()